In [1]:
import os,sys,time
 

os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PYSPARK_PYTHON"]        = sys.executable


In [2]:
from pyspark import SparkContext
from sympy import *
from drudge import *
#from gristmill import *
print("finished importing")


finished importing


In [3]:
import re

# ----------------------------------------------------------------------------
# 1) Start Spark & BCS quasiparticle drudge
# ----------------------------------------------------------------------------
ctx = SparkContext('local[*]', 'bcs_ccsd')
dr  = ReducedBCSDrudge(ctx)
dr.full_simplify = True
print("finished")

26/04/18 18:41:26 WARN Utils: Your hostname, Swarnamoys-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 10.0.0.87 instead (on interface en0)
26/04/18 18:41:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 18:41:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

finished


In [4]:
u, v = IndexedBase('u'), IndexedBase('v')
x = Symbol('x')
H00 = Symbol('H00')
P, Pdag, N = dr.names.P, dr.names.Pdag, dr.names.N
p,q,r,s,i,j,k,l = dr.names.A_dumms[:8]
P, Pdag, N = dr.names.P, dr.names.Pdag, dr.names.N
h = IndexedBase('h')
Y = IndexedBase('Y')
W = IndexedBase('W')
V = IndexedBase('V')
dr.set_symm(W, Perm([1, 0]), valence=2)
zero_term_w = [(W[p,p],0)]
P_i_dag = (u[p]*v[p]+u[p]**2*Pdag[p]
     - u[p]*v[p]*N[p]
     - v[p]**2*P[p])

P_j_dag = (
        u[q]*v[q]
      + u[q]**2 * Pdag[q]
      - u[q]*v[q] * N[q]
      - v[q]**2 * P[q]
    )

P_k_dag = (
        u[r]*v[r]
      + u[r]**2 * Pdag[r]
      - u[r]*v[r] * N[r]
      - v[r]**2 * P[r]
    )
P_i = (
        (u[p])*(v[p])
      + (u[p])**2 * P[p]
      - (u[p])*(v[p]) * N[p]
      - (v[p])**2 * Pdag[p]
    )
P_j = ((u[q])*(v[q])
      + (u[q])**2 * P[q]
      - (u[q])*(v[q]) * N[q]
      - (v[q])**2 * Pdag[q])

P_k = ((u[r])*(v[r])
      + (u[r])**2 * P[r]
      - (u[r])*(v[r]) * N[r]
      - (v[r])**2 * Pdag[r])
#hc = P_j_dag * P_i
N_i = 2*(v[p])*v[p] +((u[p])*u[p] - (v[p])*v[p])*N[p] +2*u[p]*(v[p])*Pdag[p]\
            +2*(u[p])*v[p]*P[p]
N_j = 2*(v[q])*v[q] +((u[q])*u[q] - (v[q])*v[q])*N[q] +2*u[q]*(v[q])*Pdag[q]\
            +2*(u[q])*v[q]*P[q]

N_k = 2*(v[r])*v[r] +((u[r])*u[r] - (v[r])*v[r])*N[r] +2*u[r]*(v[r])*Pdag[r]\
       +2*(u[r])*v[r]*P[r]

In [5]:
S00 = IndexedBase('S00')

S20p, S20q = IndexedBase('S20p'), IndexedBase('S20q')
S11p, S11q = IndexedBase('S11p'), IndexedBase('S11q')
S02p, S02q = IndexedBase('S02p'), IndexedBase('S02q')

S40 = IndexedBase('S40')
S31pq, S31qp = IndexedBase('S31pq'), IndexedBase('S31qp')
S22NN = IndexedBase('S22NN')
ST22pq, ST22qp = IndexedBase('ST22pq'), IndexedBase('ST22qp')
S13pq, S13qp = IndexedBase('S13pq'), IndexedBase('S13qp')
S04 = IndexedBase('S04')
dr.set_symm(S04,Perm([1,0],IDENT))
dr.set_symm(S40,Perm([1,0],IDENT)) 

In [6]:
N00 = IndexedBase('N00')

N20p, N20q = IndexedBase('N20p'), IndexedBase('N20q')
N11p, N11q = IndexedBase('N11p'), IndexedBase('N11q')
N02p, N02q = IndexedBase('N02p'), IndexedBase('N02q')

N40 = IndexedBase('N40')
N31pq, N31qp = IndexedBase('N31pq'), IndexedBase('N31qp')
N22NN = IndexedBase('N22NN')
NT22pq, NT22qp = IndexedBase('NT22pq'), IndexedBase('NT22qp')
N13pq, N13qp = IndexedBase('N13pq'), IndexedBase('N13qp')
N04 = IndexedBase('N04')
dr.set_symm(N04,Perm([1,0],IDENT))
dr.set_symm(N40,Perm([1,0],IDENT))

In [7]:
H11, H20 , H02 = IndexedBase('H11'), IndexedBase('H20'), IndexedBase('H02')
H04, H40 = IndexedBase('H04'), IndexedBase('H40')
H22, Hb22 = IndexedBase('H22'), IndexedBase('HT22')
H31, H13 = IndexedBase('H31'), IndexedBase('H13')
 
dr.set_symm(H04,Perm([1,0],IDENT))
dr.set_symm(H40,Perm([1,0],IDENT)) 

In [ ]:
#expr = H11[p]*N[p] + H02[p]*Pdag[p] +H20[p]*P[p]+H22[p,q]*N[p]*N[q]+Hb22[p,q]*Pdag[p]*P[q]\
#    +H40[p,q]*P[p]*P[q]+H04[p,q]*Pdag[p]*Pdag[q]+H13[p,q]*Pdag[p]*N[q]+H31[p,q]*N[p]*P[q]
#H_N = dr.einst(expr).simplify().merge()

In [8]:
expr = h[p]*N_i + V[p,q]*P_i_dag*P_j+ Rational(1,4)*W[p,q]*N_i*N_j
expr = dr.einst(expr).simplify().merge()
expr.latex()



'- \\sum_{p \\in A} \\left(V_{p,p} v_{p}^{4} + W_{p,p} u_{p}^{2} v_{p}^{2} - h_{p} u_{p}^{2} + h_{p} v_{p}^{2}\\right)  \\mathbf{N}_{p} - \\sum_{p \\in A} \\sum_{q \\in A} \\left(V_{p,q} u_{p}^{2} v_{q}^{2} - W_{p,q} u_{p} u_{q} v_{p} v_{q}\\right)  \\mathbf{P^\\dagger}_{p} \\mathbf{P^\\dagger}_{q}  + \\sum_{p \\in A} \\sum_{q \\in A} \\left(V_{p,q} u_{p} u_{q} v_{p} v_{q} + W_{p,q} v_{p}^{2} v_{q}^{2}\\right)   + \\sum_{p \\in A} \\sum_{q \\in A} \\left(V_{p,q} u_{p}^{2} u_{q}^{2} + V_{q,p} v_{p}^{2} v_{q}^{2} + 2 W_{p,q} u_{p} u_{q} v_{p} v_{q}\\right)  \\mathbf{P^\\dagger}_{p} \\mathbf{P}_{q}  + \\sum_{p \\in A} \\left(2 V_{p,p} u_{p} v_{p}^{3} + W_{p,p} u_{p}^{3} v_{p} - W_{p,p} u_{p} v_{p}^{3} + 2 h_{p} u_{p} v_{p}\\right)  \\mathbf{P}_{p}  + \\sum_{p \\in A} \\left(V_{p,p} v_{p}^{4} + W_{p,p} u_{p}^{2} v_{p}^{2} + 2 h_{p} v_{p}^{2}\\right)   + \\sum_{p \\in A} \\sum_{q \\in A} \\left(V_{p,q} u_{p}^{2} u_{q} v_{q} - V_{q,p} u_{q} v_{p}^{2} v_{q} + 2 W_{p,q} u_{p} v_{p} v_{q}^{2}\\

In [9]:
# HERE COMPUTING THINGS FOR PAIRING HAMILTONIAN
# WE JUST NEED <N_p> and <N_pN_q>

Pair_Np = N_i   #N_p
Pair_Np = dr.sum(Pair_Np).simplify()
Pair_Np.display()

<IPython.core.display.Math object>

In [10]:
Pair_NpNq = N_i*N_j
Pair_NpNq = dr.sum(Pair_NpNq).simplify().merge()
Pair_NpNq.display()

<IPython.core.display.Math object>

In [19]:
print(Pair_NpNq.latex())

4 v_{p}^{2}  u_{q}  v_{q}    \mathbf{P}_{q}  + \left(2 u_{p}^{2} u_{q} v_{q} - 2 u_{q} v_{p}^{2} v_{q}\right)  \mathbf{N}_{p} \mathbf{P}_{q}  + 4 u_{p}  u_{q}  v_{p}  v_{q}    \mathbf{P^\dagger}_{q} \mathbf{P}_{p}  + \left(4 \delta_{p q} u_{p}^{2} v_{p}^{2} + 4 v_{p}^{2} v_{q}^{2}\right)   + 4 u_{p}  u_{q}  v_{p}  v_{q}    \mathbf{P}_{p} \mathbf{P}_{q}  + \left(2 u_{q}^{2} v_{p}^{2} - 2 v_{p}^{2} v_{q}^{2}\right)  \mathbf{N}_{q}  + \left(u_{p}^{2} u_{q}^{2} - u_{p}^{2} v_{q}^{2} - u_{q}^{2} v_{p}^{2} + v_{p}^{2} v_{q}^{2}\right)  \mathbf{N}_{p} \mathbf{N}_{q}  + 4 u_{p}  u_{q}  v_{p}  v_{q}    \mathbf{P^\dagger}_{p} \mathbf{P^\dagger}_{q}  + \left(4 \delta_{p q} u_{p}^{3} v_{p} - 4 \delta_{p q} u_{p} v_{p}^{3} + 4 u_{p} v_{p} v_{q}^{2}\right)  \mathbf{P^\dagger}_{p} - \left(4 \delta_{p q} u_{p}^{2} v_{p}^{2} - 2 u_{p}^{2} v_{q}^{2} + 2 v_{p}^{2} v_{q}^{2}\right)  \mathbf{N}_{p}  + \left(2 u_{p} u_{q}^{2} v_{p} - 2 u_{p} v_{p} v_{q}^{2}\right)  \mathbf{N}_{q} \mathbf{P}_{p}  + \left(4 \

In [9]:
Corr =  Rational(1,4)*(N_i*N_j - N_i - N_j +1) + (P_i_dag*P_j + P_j_dag*P_i)/2
Corr = dr.sum(Corr).simplify().merge()

In [7]:
Corr.display()

<IPython.core.display.Math object>

In [8]:
print(Corr.latex())

\left(\delta_{p q} u_{p} v_{p}^{3} + \frac{u_{p} u_{q}^{2} v_{p}}{2} - \frac{u_{p} v_{p} v_{q}^{2}}{2} + u_{q} v_{p}^{2} v_{q} - \frac{u_{q} v_{q}}{2}\right)  \mathbf{P}_{q}  + \left(\frac{u_{p}^{2} u_{q} v_{q}}{2} - \frac{u_{p} u_{q}^{2} v_{p}}{2} + \frac{u_{p} v_{p} v_{q}^{2}}{2} - \frac{u_{q} v_{p}^{2} v_{q}}{2}\right)  \mathbf{N}_{p} \mathbf{P}_{q}  + \left(\frac{u_{p}^{2} u_{q}^{2}}{2} + u_{p} u_{q} v_{p} v_{q} + \frac{v_{p}^{2} v_{q}^{2}}{2}\right)  \mathbf{P^\dagger}_{q} \mathbf{P}_{p} - \left(\frac{u_{p}^{2} v_{q}^{2}}{2} - u_{p} u_{q} v_{p} v_{q} + \frac{u_{q}^{2} v_{p}^{2}}{2}\right)  \mathbf{P}_{p} \mathbf{P}_{q}  + \left(\delta_{p q} u_{p}^{2} v_{p}^{2} + \delta_{p q} v_{p}^{4} + u_{p} u_{q} v_{p} v_{q} + v_{p}^{2} v_{q}^{2} - \frac{v_{p}^{2}}{2} - \frac{v_{q}^{2}}{2} + \frac{1}{4}\right)  - \left(\frac{\delta_{p q} v_{p}^{4}}{2} + u_{p} u_{q} v_{p} v_{q} - \frac{u_{q}^{2} v_{p}^{2}}{2} + \frac{u_{q}^{2}}{4} + \frac{v_{p}^{2} v_{q}^{2}}{2} - \frac{v_{q}^{2}}{4}\right)  \mat

In [10]:
Expr = (expr + x*Corr).simplify().merge()
Expr.display()

<IPython.core.display.Math object>

In [17]:
print(Expr.latex())

\sum_{r \in A} \left(2 V_{r,r} u_{r} v_{r}^{3} + W_{r,r} u_{r}^{3} v_{r} - W_{r,r} u_{r} v_{r}^{3} + 2 h_{r} u_{r} v_{r}\right)  \mathbf{P^\dagger}_{r}  + \left(x \delta_{p q} u_{p} v_{p}^{3} + \frac{x u_{p} u_{q}^{2} v_{p}}{2} - \frac{x u_{p} v_{p} v_{q}^{2}}{2} + x u_{q} v_{p}^{2} v_{q} - \frac{x u_{q} v_{q}}{2}\right)  \mathbf{P}_{q}  + \left(\frac{x u_{p}^{2} u_{q} v_{q}}{2} - \frac{x u_{p} u_{q}^{2} v_{p}}{2} + \frac{x u_{p} v_{p} v_{q}^{2}}{2} - \frac{x u_{q} v_{p}^{2} v_{q}}{2}\right)  \mathbf{N}_{p} \mathbf{P}_{q}  + \left(\frac{x u_{p}^{2} u_{q}^{2}}{2} + x u_{p} u_{q} v_{p} v_{q} + \frac{x v_{p}^{2} v_{q}^{2}}{2}\right)  \mathbf{P^\dagger}_{q} \mathbf{P}_{p} - \sum_{r \in A} \sum_{s \in A} \left(V_{r,s} u_{s} v_{r}^{2} v_{s} - V_{s,r} u_{r}^{2} u_{s} v_{s} - W_{r,s} u_{r} v_{r} v_{s}^{2} - W_{s,r} u_{r} v_{r} v_{s}^{2}\right)  \mathbf{P}_{r} - \sum_{r \in A} \sum_{s \in A} \left(V_{r,s} u_{r} u_{s} v_{r} v_{s} + V_{s,r} u_{r} u_{s} v_{r} v_{s} - \frac{W_{r,s} u_{r}^{2} v_{s}^

In [11]:
def coeff_with_internal_sums(ts):
    """
    Strip the operator word and *only* those sums that run over
    indices appearing in the operator word. Keep any other internal sums. We need this for Integrals.
    """
    def proc(t):
        # Collect indices used in the operator word
        op_inds = set()
        for v in t.vecs:
            # depending on your Drudge version, this may be v.indices or v.inds
            for ind in v.indices:
                op_inds.add(ind)

        # Keep only sums whose index is NOT in the operator word
        new_sums = tuple((i, base) for (i, base) in t.sums if i not in op_inds)

        # Strip vecs; keep amp unchanged
        return Term(new_sums, t.amp, ())

    return ts.map(proc)

def coeff_only(ts):
    """Return a TermSum with the same sums/amp but with vecs removed."""
    # ts can be a TermSum or anything with .map over terms
    return ts.map(lambda t: Term(t.sums, t.amp, ()))   # vecs = ()

In [12]:
# Loop through each local term in H_N
for i, term in enumerate(expr.local_terms):
    print(f"--- Term {i} ---")
       # The scalar coefficient
    print("Operators (vecs):", term.vecs)   # The associated vector operators
    print()

--- Term 0 ---
Operators (vecs): (Vec('N', (p)),)

--- Term 1 ---
Operators (vecs): (Vec('P^\\dagger', (p)), Vec('P^\\dagger', (q)))

--- Term 2 ---
Operators (vecs): ()

--- Term 3 ---
Operators (vecs): (Vec('P^\\dagger', (p)), Vec('P', (q)))

--- Term 4 ---
Operators (vecs): (Vec('P', (p)),)

--- Term 5 ---
Operators (vecs): ()

--- Term 6 ---
Operators (vecs): (Vec('P^\\dagger', (p)),)

--- Term 7 ---
Operators (vecs): (Vec('P', (p)),)

--- Term 8 ---
Operators (vecs): (Vec('N', (p)),)

--- Term 9 ---
Operators (vecs): (Vec('P', (p)), Vec('P', (q)))

--- Term 10 ---
Operators (vecs): (Vec('P^\\dagger', (p)), Vec('N', (q)))

--- Term 11 ---
Operators (vecs): (Vec('N', (p)), Vec('P', (q)))

--- Term 12 ---
Operators (vecs): (Vec('N', (p)), Vec('N', (q)))

--- Term 13 ---
Operators (vecs): (Vec('P^\\dagger', (p)),)



In [13]:
def get_PdagPdagP(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
    # Normal-ordered triple: P† P† P
    return labels == ['P^\\dagger', 'P^\\dagger','P']

def get_PdagPP(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger', 'P','P']

def get_NNN(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['N', 'N','N']


def get_PPP(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P', 'P','P']


def get_PdagPdagN(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger', 'P^\\dagger','N']


def get_NNP(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['N', 'N','P']


def get_NPP(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['N', 'P','P']


def get_PdagPdagPdag(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger','P^\\dagger','P^\\dagger']


def get_PdagNN(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger', 'N','N']


def get_PdagNP(term):
    vecs = term.vecs
    # Must be exactly 3 operators
    if len(vecs) != 3:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger', 'N','P']

def get_Pdag(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 1:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger']

def get_P(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 1:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P']

def get_N(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 1:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['N']

def get_NN(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 2:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['N','N']

def get_NP(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 2:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['N','P']

def get_PP(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 2:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P','P']

def get_PdagPdag(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 2:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger','P^\\dagger']

def get_PdagN(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 2:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger','N']

def get_PdagP(term):
    vecs = term.vecs
    # Must be exactly 1 operators
    if len(vecs) != 2:
        return False
    labels = [v.label for v in vecs]
     
    return labels == ['P^\\dagger','P']



In [14]:
def drop_terms_containing_P(expr):

    ''' Assuming expr is normal_ordered '''
    kept = expr.simplify().filter(lambda term: all(v.label != 'P' for v in term.vecs))
     
    #return Tensor(dr,kept).simplify().merge()
    return kept

def keep_exactly_k_Pdag(expr, k ,pdag_label='P^\\dagger'):

    """
    Keep only terms with exactly k creators P^\\dagger in the operator string.
    Assumes expr has already had all P (annihilators) filtered out and is normal ordered.
    """
    def nPdag(term):
        return sum(1 for v in term.vecs if v.label == pdag_label)

    kept = expr.simplify().filter(lambda term: nPdag(term) == k)
    #return Tensor(dr, kept).simplify().merge()
    return kept
    

In [11]:
tensor0_ = Expr.filter(lambda term: len(term.vecs) == 0)
#tensor0_ = dr.einst(tensor0_).simplify().merge().merge()
tensor0_ = tensor0_.simplify().merge().merge()
 
tensor0_.display()


<IPython.core.display.Math object>

In [12]:
SOO_dr = tensor0_

In [13]:
SOO_dr.display()
print(SOO_dr.latex())

\sum_{r \in A} \sum_{s \in A} \left(V_{r,s} u_{r} u_{s} v_{r} v_{s} + W_{r,s} v_{r}^{2} v_{s}^{2}\right)   + \sum_{r \in A} \left(V_{r,r} v_{r}^{4} + W_{r,r} u_{r}^{2} v_{r}^{2} + 2 h_{r} v_{r}^{2}\right)   + \left(x \delta_{p q} u_{p}^{2} v_{p}^{2} + x \delta_{p q} v_{p}^{4} + x u_{p} u_{q} v_{p} v_{q} + x v_{p}^{2} v_{q}^{2} - \frac{x v_{p}^{2}}{2} - \frac{x v_{q}^{2}}{2} + \frac{x}{4}\right) 


In [27]:
F_pq_u = (SOO_dr.diff(u[j])).simplify()
F_pq_u.display()

<IPython.core.display.Math object>

In [30]:
F_pq_v = (SOO_dr.diff(v[j])).simplify()
F_pq_v.display()

<IPython.core.display.Math object>

In [32]:
F_pq = (F_pq_u*(-v[j]) + F_pq_v*u[j]).simplify().merge()
F_pq.display()

<IPython.core.display.Math object>

In [33]:
print(F_pq.latex())

\left(2 x \delta_{p p_{1}} \delta_{p q} u_{p}^{3} v_{p} + 2 x \delta_{p p_{1}} \delta_{p q} u_{p} v_{p}^{3} + x \delta_{p p_{1}} u_{p}^{2} u_{q} v_{q} + 2 x \delta_{p p_{1}} u_{p} v_{p} v_{q}^{2} - x \delta_{p p_{1}} u_{p} v_{p} - x \delta_{p p_{1}} u_{q} v_{p}^{2} v_{q} + x \delta_{p_{1} q} u_{p_{1}}^{2} u_{p} v_{p} + 2 x \delta_{p_{1} q} u_{p_{1}} v_{p_{1}} v_{p}^{2} - x \delta_{p_{1} q} u_{p_{1}} v_{p_{1}} - x \delta_{p_{1} q} u_{p} v_{p_{1}}^{2} v_{p} + 4 V_{p_{1},p_{1}} u_{p_{1}} v_{p_{1}}^{3} + 2 W_{p_{1},p_{1}} u_{p_{1}}^{3} v_{p_{1}} - 2 W_{p_{1},p_{1}} u_{p_{1}} v_{p_{1}}^{3} + 4 h_{p_{1}} u_{p_{1}} v_{p_{1}}\right)   + \sum_{r \in A} \left(V_{p_{1},r} u_{p_{1}}^{2} u_{r} v_{r} - V_{p_{1},r} u_{r} v_{p_{1}}^{2} v_{r} + V_{r,p_{1}} u_{p_{1}}^{2} u_{r} v_{r} - V_{r,p_{1}} u_{r} v_{p_{1}}^{2} v_{r} + 2 W_{p_{1},r} u_{p_{1}} v_{p_{1}} v_{r}^{2} + 2 W_{r,p_{1}} u_{p_{1}} v_{p_{1}} v_{r}^{2}\right) 


In [23]:
#UNPERTURBED BCS ENERGY GRADIENT dE_BCS/d theta 
tensor0_ = Expr.filter(lambda term: len(term.vecs) == 0)
#tensor0_ = dr.einst(tensor0_).simplify().merge().merge()
tensor0_ = tensor0_.simplify().merge().merge()
dA00 = tensor0_
tensor0_.display()

A00_u = (dA00.diff(u[r])).simplify()
A00_u.display()
A00_v = (dA00.diff(v[r])).simplify()

A00 = (A00_u*(-v[r]) + A00_v*(u[r])).simplify().merge()
A00 = A00.subst_all(zero_term_w)
print(A00.latex())
A00.display()

4 v_{r}^{3}  u_{r}  V_{r,r}   + 4 h_{r}  u_{r}  v_{r}  -  x \delta_{p r} u_{p}  v_{p}  -  x \delta_{q r} u_{q}  v_{q}   + x \delta_{p r} u_{p}^{2}  u_{q}  v_{q}   + x \delta_{q r} u_{q}^{2}  u_{p}  v_{p}  -  x \delta_{p r} v_{p}^{2}  u_{q}  v_{q}  -  x \delta_{q r} v_{q}^{2}  u_{p}  v_{p}   + 2 x \delta_{p q} \delta_{p r} u_{p}^{3}  v_{p}   + 2 x \delta_{q r} v_{p}^{2}  u_{q}  v_{q}   + 2 x \delta_{p q} \delta_{p r} v_{p}^{3}  u_{p}   + 2 x \delta_{p r} v_{q}^{2}  u_{p}  v_{p}   + \sum_{s \in A} u_{r}^{2}  u_{s}  v_{s}  V_{r,s}   + \sum_{s \in A} u_{r}^{2}  u_{s}  v_{s}  V_{s,r}  - \sum_{s \in A} v_{r}^{2}  u_{s}  v_{s}  V_{r,s}  - \sum_{s \in A} v_{r}^{2}  u_{s}  v_{s}  V_{s,r}   + \sum_{s \in A} 4 v_{s}^{2}  u_{r}  v_{r}  W_{s,r} 


<IPython.core.display.Math object>

In [22]:
#THE HESSIAN OF THE UNPERTURBED BCS ENERGY =A0_rs

d2A00_u = (A00.diff(u[s])).simplify()
 
d2A00_v = (A00.diff(v[s])).simplify()

d2A00 = (d2A00_u*(-v[s]) + d2A00_v*(u[s])).simplify().merge()
print(d2A00.latex())

- \sum_{p_{0} \in A} \left(4 \delta_{r s} V_{p_{0},r} u_{p_{0}} u_{r} v_{p_{0}} v_{r} + 4 \delta_{r s} V_{r,p_{0}} u_{p_{0}} u_{r} v_{p_{0}} v_{r} - 4 \delta_{r s} W_{p_{0},r} u_{r}^{2} v_{p_{0}}^{2} + 4 \delta_{r s} W_{p_{0},r} v_{p_{0}}^{2} v_{r}^{2}\right)   + \left(2 x \delta_{p q} \delta_{p r} \delta_{p s} u_{p}^{4} - 2 x \delta_{p q} \delta_{p r} \delta_{p s} v_{p}^{4} + 2 x \delta_{p r} \delta_{p s} u_{p}^{2} v_{q}^{2} - x \delta_{p r} \delta_{p s} u_{p}^{2} - 4 x \delta_{p r} \delta_{p s} u_{p} u_{q} v_{p} v_{q} - 2 x \delta_{p r} \delta_{p s} v_{p}^{2} v_{q}^{2} + x \delta_{p r} \delta_{p s} v_{p}^{2} + x \delta_{p r} \delta_{q s} u_{p}^{2} u_{q}^{2} - x \delta_{p r} \delta_{q s} u_{p}^{2} v_{q}^{2} + 4 x \delta_{p r} \delta_{q s} u_{p} u_{q} v_{p} v_{q} - x \delta_{p r} \delta_{q s} u_{q}^{2} v_{p}^{2} + x \delta_{p r} \delta_{q s} v_{p}^{2} v_{q}^{2} + x \delta_{p s} \delta_{q r} u_{p}^{2} u_{q}^{2} - x \delta_{p s} \delta_{q r} u_{p}^{2} v_{q}^{2} + 4 x \delta_{p s} \delta_

In [19]:
tensor0_.display()

<IPython.core.display.Math object>

In [24]:
#PERTURBED BCS ENERGY GRADIENT dE_(xH1)/d theta 
Corr_tensor0_ = Corr.filter(lambda term: len(term.vecs) == 0)
#tensor0_ = dr.einst(tensor0_).simplify().merge().merge()
Corr_tensor0_ = Corr_tensor0_.simplify().merge().merge()
dB00 = Corr_tensor0_
Corr_tensor0_.display()

B00_u = (dB00.diff(u[r])).simplify()
B00_u.display()
B00_v = (dB00.diff(v[r])).simplify()

B00 = (B00_u*(-v[r]) + B00_v*(u[r])).simplify().merge()
print(B00.latex())
#A00.display()

\left(2 \delta_{p q} \delta_{p r} u_{p}^{3} v_{p} + 2 \delta_{p q} \delta_{p r} u_{p} v_{p}^{3} + \delta_{p r} u_{p}^{2} u_{q} v_{q} + 2 \delta_{p r} u_{p} v_{p} v_{q}^{2} - \delta_{p r} u_{p} v_{p} - \delta_{p r} u_{q} v_{p}^{2} v_{q} + \delta_{q r} u_{p} u_{q}^{2} v_{p} - \delta_{q r} u_{p} v_{p} v_{q}^{2} + 2 \delta_{q r} u_{q} v_{p}^{2} v_{q} - \delta_{q r} u_{q} v_{q}\right) 


In [25]:
d2B00_u = (B00.diff(u[s])).simplify()
d2B00_v = (B00.diff(v[s])).simplify()

d2B00 = (d2B00_u*(-v[s]) + d2B00_v*(u[s])).simplify().merge()
print(d2B00.latex())

\left(2 \delta_{p q} \delta_{p r} \delta_{p s} u_{p}^{4} - 2 \delta_{p q} \delta_{p r} \delta_{p s} v_{p}^{4} + 2 \delta_{p r} \delta_{p s} u_{p}^{2} v_{q}^{2} - \delta_{p r} \delta_{p s} u_{p}^{2} - 4 \delta_{p r} \delta_{p s} u_{p} u_{q} v_{p} v_{q} - 2 \delta_{p r} \delta_{p s} v_{p}^{2} v_{q}^{2} + \delta_{p r} \delta_{p s} v_{p}^{2} + \delta_{p r} \delta_{q s} u_{p}^{2} u_{q}^{2} - \delta_{p r} \delta_{q s} u_{p}^{2} v_{q}^{2} + 4 \delta_{p r} \delta_{q s} u_{p} u_{q} v_{p} v_{q} - \delta_{p r} \delta_{q s} u_{q}^{2} v_{p}^{2} + \delta_{p r} \delta_{q s} v_{p}^{2} v_{q}^{2} + \delta_{p s} \delta_{q r} u_{p}^{2} u_{q}^{2} - \delta_{p s} \delta_{q r} u_{p}^{2} v_{q}^{2} + 4 \delta_{p s} \delta_{q r} u_{p} u_{q} v_{p} v_{q} - \delta_{p s} \delta_{q r} u_{q}^{2} v_{p}^{2} + \delta_{p s} \delta_{q r} v_{p}^{2} v_{q}^{2} - 4 \delta_{q r} \delta_{q s} u_{p} u_{q} v_{p} v_{q} + 2 \delta_{q r} \delta_{q s} u_{q}^{2} v_{p}^{2} - \delta_{q r} \delta_{q s} u_{q}^{2} - 2 \delta_{q r} \delta_{q

In [56]:
tensor0_.display()

<IPython.core.display.Math object>

In [ ]:
tensor1_ = Expr.filter(lambda term: len(term.vecs) == 1)

#tensor1_ = dr.einst(tensor1_).simplify().merge().merge()
tensor1_ = tensor1_.simplify().merge().merge()
tensor1_.display()

In [ ]:
Pdag_terms = tensor1_.terms.filter(get_Pdag)
#H201_dr = Tensor(dr,  coeff_with_internal_sums(pdagpdagp_terms)).simplify().merge()
S02 = Tensor(dr,   coeff_only(Pdag_terms)).simplify().merge()


In [ ]:
S02.display()

In [ ]:
S02.latex()

In [ ]:
P_terms = tensor1_.terms.filter(get_P)
#H201_dr = Tensor(dr,  coeff_with_internal_sums(pdagpdagp_terms)).simplify().merge()
S20 = Tensor(dr,   coeff_only(P_terms)).simplify().merge()

In [ ]:
tensor2_ = Corr.filter(lambda term: len(term.vecs) == 2)

#tensor1_ = dr.einst(tensor1_).simplify().merge().merge()
tensor2_ = tensor2_.simplify().merge().merge()
tensor2_.display()

In [9]:
Sexpr = (
    S00[p,q]
    + S02p[p, q] * Pdag[p]
    + S02q[p, q] * Pdag[q]
    + S11p[p, q] * N[p]
    + S11q[p, q] * N[q]
    + S20p[p, q] * P[p]
    + S20q[p, q] * P[q]
    + S04[p, q] * Pdag[p] * Pdag[q]
    + S13pq[p, q] * Pdag[p] * N[q]
    + S13qp[p, q] * Pdag[q] * N[p]
    + S22NN[p, q] * N[p] * N[q]
    + ST22pq[p, q] * Pdag[p] * P[q]
    + ST22qp[p, q] * Pdag[q] * P[p]
    + S31pq[p, q] * N[p] * P[q]
    + S31qp[p, q] * N[q] * P[p]
    + S40[p, q] * P[p] * P[q]
)
Sexpr = dr.sum(Sexpr).simplify()

#Hexpr = H00 + H11[p]*N[p] + H02[p]*Pdag[p] +H20[p]*P[p]+H22[p,q]*N[p]*N[q]+Hb22[p,q]*Pdag[p]*P[q]\
#    +H40[p,q]*P[p]*P[q]+H04[p,q]*Pdag[p]*Pdag[q]+H13[p,q]*Pdag[p]*N[q]+H31[p,q]*N[p]*P[q]
#Hexpr = dr.einst(Hexpr).simplify().merge()

#expr = (Hexpr + x*Sexpr).simplify().merge()
 

 

In [15]:
Pair_NpNq_expr = (
    N00[p,q]
    + N02p[p, q] * Pdag[p]
    + N02q[p, q] * Pdag[q]
    + N11p[p, q] * N[p]
    + N11q[p, q] * N[q]
    + N20p[p, q] * P[p]
    + N20q[p, q] * P[q]
    + N04[p, q] * Pdag[p] * Pdag[q]
    + N13pq[p, q] * Pdag[p] * N[q]
    + N13qp[p, q] * Pdag[q] * N[p]
    + N22NN[p, q] * N[p] * N[q]
    + NT22pq[p, q] * Pdag[p] * P[q]
    + NT22qp[p, q] * Pdag[q] * P[p]
    + N31pq[p, q] * N[p] * P[q]
    + N31qp[p, q] * N[q] * P[p]
    + N40[p, q] * P[p] * P[q]
)
Pair_NpNq_expr = dr.sum(Pair_NpNq_expr).simplify()

In [29]:
print(Pair_NpNq_expr.latex())
Pair_NpNq_expr.display()

N31pq_{p,q}    \mathbf{N}_{p} \mathbf{P}_{q}  + N20q_{p,q}    \mathbf{P}_{q}  + NT22qp_{p,q}    \mathbf{P^\dagger}_{q} \mathbf{P}_{p}  + N^{(40)}_{p,q}    \mathbf{P}_{p} \mathbf{P}_{q}  + N11q_{p,q}    \mathbf{N}_{q}  + N22NN_{p,q}    \mathbf{N}_{p} \mathbf{N}_{q}  + N^{(00)}_{p,q}   + N^{(04)}_{p,q}    \mathbf{P^\dagger}_{p} \mathbf{P^\dagger}_{q}  + N02p_{p,q}    \mathbf{P^\dagger}_{p}  + N31qp_{p,q}    \mathbf{N}_{q} \mathbf{P}_{p}  + N11p_{p,q}    \mathbf{N}_{p}  + N20p_{p,q}    \mathbf{P}_{p}  + N13pq_{p,q}    \mathbf{P^\dagger}_{p} \mathbf{N}_{q}  + N13qp_{p,q}    \mathbf{P^\dagger}_{q} \mathbf{N}_{p}  + NT22pq_{p,q}    \mathbf{P^\dagger}_{p} \mathbf{P}_{q}  + N02q_{p,q}    \mathbf{P^\dagger}_{q}


<IPython.core.display.Math object>

In [13]:
print(Sexpr.latex())

S31pq_{p,q}    \mathbf{N}_{p} \mathbf{P}_{q}  + S20q_{p,q}    \mathbf{P}_{q}  + ST22qp_{p,q}    \mathbf{P^\dagger}_{q} \mathbf{P}_{p}  + S^{(40)}_{p,q}    \mathbf{P}_{p} \mathbf{P}_{q}  + S11q_{p,q}    \mathbf{N}_{q}  + S22NN_{p,q}    \mathbf{N}_{p} \mathbf{N}_{q}  + S^{(00)}_{p,q}   + S^{(04)}_{p,q}    \mathbf{P^\dagger}_{p} \mathbf{P^\dagger}_{q}  + S02p_{p,q}    \mathbf{P^\dagger}_{p}  + S31qp_{p,q}    \mathbf{N}_{q} \mathbf{P}_{p}  + S11p_{p,q}    \mathbf{N}_{p}  + S20p_{p,q}    \mathbf{P}_{p}  + S13pq_{p,q}    \mathbf{P^\dagger}_{p} \mathbf{N}_{q}  + S13qp_{p,q}    \mathbf{P^\dagger}_{q} \mathbf{N}_{p}  + ST22pq_{p,q}    \mathbf{P^\dagger}_{p} \mathbf{P}_{q}  + S02q_{p,q}    \mathbf{P^\dagger}_{q}


In [16]:
t1, t2, t3,t4 = IndexedBase('t1'), IndexedBase('t2'), IndexedBase('t3'), IndexedBase('t4')

z1, z2 = IndexedBase('z1'), IndexedBase('z2')
z3, z4 = IndexedBase('z3'), IndexedBase('z4')

Z1 = dr.einst(z1[p]*P[p])
Z2 = dr.einst(Rational(1,2)*z2[p,q]*P[p]*P[q])
Z3 = dr.einst(Rational(1,6)*z3[p,q,r]*P[p]*P[q]*P[r])
Z4 = dr.einst(Rational(1,24)*z4[p,q,r,s]*P[p]*P[q]*P[r]*P[s])
dr.set_symm(z2, Perm([1, 0],IDENT), valence=2)
dr.set_symm(z3, Perm([1,0,2],IDENT), Perm([0,2,1],IDENT), valence=3) 
dr.set_symm(z4,Perm([1,0,2,3],IDENT),Perm([0,1,3,2],IDENT),Perm([0,2,1,3],IDENT))

T1 = dr.einst(t1[p] * Pdag[p])
T2 = dr.einst(Rational(1, 2) * t2[p,q] *Pdag[p]*Pdag[q])
T3 = dr.einst(Rational(1,6)*t3[p,q,r]*Pdag[p]*Pdag[q]*Pdag[r])
T4 = dr.einst(Rational(1,24)*t4[p,q,r,s]*Pdag[p]*Pdag[q]*Pdag[r]*Pdag[s])
dr.set_symm(t2, Perm([1, 0],IDENT), valence=2)
dr.set_symm(t3, Perm([1,0,2],IDENT), Perm([0,2,1],IDENT), valence=3)
dr.set_symm(t4,Perm([1,0,2,3],IDENT),Perm([0,1,3,2],IDENT),Perm([0,2,1,3],IDENT))

Z =  Z1+Z2 +Z3+ Z4
cluster = T1+ T2 +T3+ T4
cluster.display()

<IPython.core.display.Math object>

In [17]:
zero_term = [(t2[p,p],0),(t4[p,p,p,p],0),(t3[p,p,p],0),(t3[p,p,q],0),(t3[p,q,p],0),(z3[p,p,p],0),(z3[q,p,p],0),(z3[p,p,q],0),(z2[p,p],0),(z4[p,p,p,p],0),
    (t3[q,p,p],0),(t3[p,q,p],0),(t4[p,p,r,s],0),(t4[r,p,p,s],0),(t4[r,s,p,p],0),(t4[p,r,p,s],0),(t4[p,r,s,p],0),(t4[r,p,s,p],0),
            (z4[p,p,r,s],0),(z4[r,p,p,s],0),(z4[r,s,p,p],0),(z4[p,r,p,s],0),(z4[p,r,s,p],0),(z4[r,p,s,p],0),(S40[p,p], 0), (S04[p,p], 0),
             (N40[p,p],N04[p,p])]
                                                                                                                                        

In [10]:
t0 =time.time()
SpSqbar = Sexpr #change it to expr,Sexpr accordingly
curr = Sexpr
fact = 1
for n in range(6):                              # up to 4-fold for 2-body H
    comm = (curr | cluster).simplify()          # ad_T^{n+1}(H_N)
    fact *= (n + 1)                             # 1!,2!,3!,4!
    SpSqbar += comm / fact                         # add 1/(n+1)! term
    curr = comm    
print("SpSqbar evaluation done ", time.time()-t0)


[Stage 171:==============================================>     (258 + 12) / 288]

SpSqbar evaluation done  15.92759108543396


In [18]:
# CALCULATE Hbar FOR PAIRING

t0 =time.time()
Npbar = Pair_Np #change it to expr,Sexpr accordingly
curr = Pair_Np
fact = 1
for n in range(6):                              # up to 4-fold for 2-body H
    comm = (curr | cluster).simplify()          # ad_T^{n+1}(H_N)
    fact *= (n + 1)                             # 1!,2!,3!,4!
    Npbar += comm / fact                         # add 1/(n+1)! term
    curr = comm    
print("SpSqbar evaluation done ", time.time()-t0)

[Stage 207:================================================>   (542 + 12) / 576]

SpSqbar evaluation done  21.08932876586914


In [19]:
# CALCULATE NpNqbar FOR PAIRING

t0 =time.time()
NpNqbar = Pair_NpNq_expr #change it to expr,Sexpr accordingly
curr = Pair_NpNq_expr
fact = 1
for n in range(6):                              # up to 4-fold for 2-body H
    comm = (curr | cluster).simplify()          # ad_T^{n+1}(H_N)
    fact *= (n + 1)                             # 1!,2!,3!,4!
    NpNqbar += comm / fact                         # add 1/(n+1)! term
    curr = comm    
print("SpSqbar evaluation done ", time.time()-t0)
NpNqbar = NpNqbar.simplify()

SpSqbar evaluation done  89.78770089149475


In [11]:
SpSqbar = SpSqbar.simplify()

In [12]:
t0 = time.time()
     
Ex_SpSq =  ((1+Z)*(SpSqbar)).simplify().eval_vev().simplify()
print("finished ", time.time()-t0)

finished  108.39175486564636


In [26]:
# <Np> CCSD
t0 = time.time()
     
Ex_Np =  ((1+Z)*(Npbar)).simplify().eval_vev().simplify()
print("finished ", time.time()-t0)

finished  19.128875017166138


In [22]:
# <NpNq> for CCSD

t0 = time.time()
     
Ex_NpNq =  ((1+Z)*(NpNqbar)).simplify().eval_vev().simplify()
print("finished ", time.time()-t0)

finished  108.67591500282288


In [27]:
Ex_NpNq = Ex_NpNq.subst_all(zero_term)
Ex_Np = Ex_Np.subst_all(zero_term)

In [ ]:
'''Defining CC Lagrangian
L = <0|(1+Z) bar{H}|0> and dL/dt = 0 is the Z equations. dL/dt = <0|(1+Z)[bar{H},Q_{mu}]|0>
'''
t0 = time.time()

SpSqbar = SpSqbar.simplify()
SpSqbar_noP =  drop_terms_containing_P(SpSqbar)            

# 2) Simplify the commutator itself (usually shrinks a lot)
 

C0 = keep_exactly_k_Pdag(SpSqbar_noP, 0)
C1 = keep_exactly_k_Pdag(SpSqbar_noP, 1)
C2 = keep_exactly_k_Pdag(SpSqbar_noP, 2)
C3 = keep_exactly_k_Pdag(SpSqbar_noP, 3)
C4 = keep_exactly_k_Pdag(SpSqbar_noP, 4)

# 3) Use linearity: <0|(1+Z) C|0> = <0|C|0> + <0|Z C|0>
term0 = C0.eval_vev().simplify()

term1 = (Z1 * C1).simplify().eval_vev().simplify()
term2 = (Z2 * C2).simplify().eval_vev().simplify()
term3 = (Z3 * C3).simplify().eval_vev().simplify()
term4 = (Z4 * C4).simplify().eval_vev().simplify()

Ex_SpSq = (term0 + term1 + term2 + term3 + term4).simplify()

#termZ = (Z * C).simplify().eval_vev().simplify() 
#termZ = (Z * C).normal_order().eval_vev().simplify() 
#dL_dt_1 = (term0 + termZ).simplify().merge() 
 
#dL_dt_1 = ((1+Z)*(Hbar|Pdag[p])).simplify().eval_vev().simplify().merge()
print("finish time: ",time.time()-t0 )

In [13]:
Ex_SpSq  = Ex_SpSq.subst_all(zero_term)

In [22]:
# CCSDTQ <Np>

t0 = time.time()


Npbar = Npbar.simplify()
Npbar_noP =  drop_terms_containing_P(Npbar)            

# 2) Simplify the commutator itself (usually shrinks a lot)
 

C0 = keep_exactly_k_Pdag(Npbar_noP, 0)
C1 = keep_exactly_k_Pdag(Npbar_noP, 1)
C2 = keep_exactly_k_Pdag(Npbar_noP, 2)
C3 = keep_exactly_k_Pdag(Npbar_noP, 3)
C4 = keep_exactly_k_Pdag(Npbar_noP, 4)

# 3) Use linearity: <0|(1+Z) C|0> = <0|C|0> + <0|Z C|0>
term0 = C0.eval_vev().simplify()

term1 = (Z1 * C1).simplify().eval_vev().simplify()
term2 = (Z2 * C2).simplify().eval_vev().simplify()
term3 = (Z3 * C3).simplify().eval_vev().simplify()
term4 = (Z4 * C4).simplify().eval_vev().simplify()

Ex_Np = (term0 + term1 + term2 + term3 + term4).simplify()

#termZ = (Z * C).simplify().eval_vev().simplify() 
#termZ = (Z * C).normal_order().eval_vev().simplify() 
#dL_dt_1 = (term0 + termZ).simplify().merge() 
 
#dL_dt_1 = ((1+Z)*(Hbar|Pdag[p])).simplify().eval_vev().simplify().merge()
print("finish time: ",time.time()-t0 )

[Stage 2876:=============================================>        (10 + 2) / 12]

finish time:  519.9690160751343


In [ ]:
# CCSDTQ <NpNq>

t0 = time.time()


NpNqbar = NpNqbar.simplify()
NpNqbar_noP =  drop_terms_containing_P(NpNqbar)            

# 2) Simplify the commutator itself (usually shrinks a lot)
 

C0 = keep_exactly_k_Pdag(NpNqbar_noP, 0)
C1 = keep_exactly_k_Pdag(NpNqbar_noP, 1)
C2 = keep_exactly_k_Pdag(NpNqbar_noP, 2)
C3 = keep_exactly_k_Pdag(NpNqbar_noP, 3)
C4 = keep_exactly_k_Pdag(NpNqbar_noP, 4)

# 3) Use linearity: <0|(1+Z) C|0> = <0|C|0> + <0|Z C|0>
term0 = C0.eval_vev().simplify()

term1 = (Z1 * C1).simplify().eval_vev().simplify()
term2 = (Z2 * C2).simplify().eval_vev().simplify()
term3 = (Z3 * C3).simplify().eval_vev().simplify()
term4 = (Z4 * C4).simplify().eval_vev().simplify()

Ex_NpNq = (term0 + term1 + term2 + term3 + term4).simplify()

#termZ = (Z * C).simplify().eval_vev().simplify() 
#termZ = (Z * C).normal_order().eval_vev().simplify() 
#dL_dt_1 = (term0 + termZ).simplify().merge() 
 
#dL_dt_1 = ((1+Z)*(Hbar|Pdag[p])).simplify().eval_vev().simplify().merge()
print("finish time: ",time.time()-t0 )

In [25]:
#Ex_NpNq = Ex_NpNq.subst_all(zero_term)
Ex_Np = Ex_Np.subst_all(zero_term)

In [14]:
import time
from drudge import Tensor
from gristmill import optimize, get_flop_cost
from gristmill.generate import FortranPrinter
from sympy import IndexedBase
from sympy.printing.fortran import FCodePrinter

# Patch SymPy's Fortran printer:
# KroneckerDelta(i,j) -> deltaf(i,j)
def _print_KroneckerDelta(self, expr):
    i, j = expr.args
    return f"deltaf({self._print(i)}, {self._print(j)})"

FCodePrinter._print_KroneckerDelta = _print_KroneckerDelta

t0 = time.time()
SpSq = IndexedBase('SpSq')

expr1 = dr.define(SpSq[p,q], Ex_SpSq)
org_exp = [expr1]
 
print("Original Cost:", get_flop_cost(org_exp))
eval_equ0 = optimize(
    [expr1],
    interm_fmt='tau{}',
    drop_cutoff=2
)

print(type(eval_equ0))
 
print("Optimized cost:", get_flop_cost(eval_equ0))

 
fort = FortranPrinter()
code = fort.doprint(eval_equ0)
#print(code)


delta_func = """
pure double precision function deltaf(p, q)
  implicit none
  integer, intent(in) :: p, q

  if (p == q) then
    deltaf = 1.0d0
  else
    deltaf = 0.0d0
  end if

end function deltaf
"""

with open("BCS_CCSD_SpSq.f90", "w") as fp:
    fp.write(delta_func)
    fp.write("\n\n")
    fp.write(code)

print("Printed!", time.time() - t0)


Original Cost: 4*na**4 + 90*na**3 + 146*na**2
<class 'list'>
Optimized cost: 4*na**3 + 86*na**2 + 16*na
Printed! 13.392551183700562


In [27]:
import time
from drudge import Tensor
from gristmill import optimize, get_flop_cost
from gristmill.generate import FortranPrinter
from sympy import IndexedBase
from sympy.printing.fortran import FCodePrinter

# Patch SymPy's Fortran printer:
# KroneckerDelta(i,j) -> deltaf(i,j)
def _print_KroneckerDelta(self, expr):
    i, j = expr.args
    return f"deltaf({self._print(i)}, {self._print(j)})"

FCodePrinter._print_KroneckerDelta = _print_KroneckerDelta

t0 = time.time()
Agrad = IndexedBase('A')
Bgrad = IndexedBase('B')

expr1 = dr.define(Agrad[r,s], d2A00)
expr2 = dr.define(Bgrad[r,s], B00)
org_exp = [expr1,expr2]
 
print("Original Cost:", get_flop_cost(org_exp))
eval_equ0 = optimize(
    [expr1,expr2],
    interm_fmt='tau{}',
    drop_cutoff=2
)


print(type(eval_equ0))
 
print("Optimized cost:", get_flop_cost(eval_equ0))

 
fort = FortranPrinter()
code = fort.doprint(eval_equ0)
#print(code)


delta_func = """
pure double precision function deltaf(p, q)
  implicit none
  integer, intent(in) :: p, q

  if (p == q) then
    deltaf = 1.0d0
  else
    deltaf = 0.0d0
  end if

end function deltaf
"""

with open("Orbital_relax.f90", "w") as fp:
    fp.write(delta_func)
    fp.write("\n\n")
    fp.write(code)

print("Printed!", time.time() - t0)


Original Cost: na**3 + na**2


AssertionError: 

In [26]:
import time
from drudge import Tensor
from gristmill import optimize, get_flop_cost
from gristmill.generate import FortranPrinter
from sympy import IndexedBase
from sympy.printing.fortran import FCodePrinter

# Patch SymPy's Fortran printer:
# KroneckerDelta(i,j) -> deltaf(i,j)
def _print_KroneckerDelta(self, expr):
    i, j = expr.args
    return f"deltaf({self._print(i)}, {self._print(j)})"

FCodePrinter._print_KroneckerDelta = _print_KroneckerDelta

t0 = time.time()
Agrad = IndexedBase('A')
Bgrad = IndexedBase('B')

expr1 = dr.define(Agrad[r,s], d2A00)
expr2 = dr.define(Bgrad[r,s], B00)
org_exp = [expr1,expr2]
 
print("Original Cost:", get_flop_cost(org_exp))
eval_equ0 = optimize(
    [expr1,expr2],
    interm_fmt='tau{}',
    drop_cutoff=2
)

print(type(eval_equ0))
 
print("Optimized cost:", get_flop_cost(eval_equ0))

fort = FortranPrinter()
 

decls, evals_list = fort.print_decl_eval(eval_equ0, decl_type='real', explicit_bounds=False)

code = "\n".join(decls) + "\n\n" + "\n\n".join(evals_list)
delta_func = """
pure double precision function deltaf(p, q)
  implicit none
  integer, intent(in) :: p, q

  if (p == q) then
    deltaf = 1.0d0
  else
    deltaf = 0.0d0
  end if

end function deltaf
"""

with open("Orbital_relax.f90", "w") as fp:
    fp.write(delta_func)
    fp.write("\n\n")
    fp.write(code)

print("Printed!", time.time() - t0)


Original Cost: na**3 + na**2


/Users/swarnamoyghosh/anaconda3/envs/drudge_fix/lib/python3.9/site-packages/gristmill/optimize.py:1984: UserWarning: Internal deficiency: summation intermediate may not be fully canonicalized
  warnings.warn(


AssertionError: 

In [28]:
import time
from drudge import Tensor
from gristmill import optimize, get_flop_cost
from gristmill.generate import FortranPrinter
from sympy import IndexedBase
from sympy.printing.fortran import FCodePrinter

# Patch SymPy's Fortran printer:
# KroneckerDelta(i,j) -> deltaf(i,j)
def _print_KroneckerDelta(self, expr):
    i, j = expr.args
    return f"deltaf({self._print(i)}, {self._print(j)})"

FCodePrinter._print_KroneckerDelta = _print_KroneckerDelta

t0 = time.time()
NpNq = IndexedBase('NpNq')

expr1 = dr.define(NpNq[p,q], Ex_NpNq)
org_exp = [expr1]
 
print("Original Cost:", get_flop_cost(org_exp))
eval_equ0 = optimize(
    [expr1],
    interm_fmt='tau{}',
    drop_cutoff=2
)

print(type(eval_equ0))
 
print("Optimized cost:", get_flop_cost(eval_equ0))

 
fort = FortranPrinter()
code = fort.doprint(eval_equ0)
#print(code)


delta_func = """
pure double precision function deltaf(p, q)
  implicit none
  integer, intent(in) :: p, q

  if (p == q) then
    deltaf = 1.0d0
  else
    deltaf = 0.0d0
  end if

end function deltaf
"""

with open("BCS_CCSD_NpNq.f90", "w") as fp:
    fp.write(delta_func)
    fp.write("\n\n")
    fp.write(code)

print("Printed!", time.time() - t0)


Original Cost: 9*na**4 + 100*na**3 + 156*na**2


<class 'list'>
Optimized cost: 4*na**3 + 93*na**2 + 29*na
Printed! 13.015245914459229


In [26]:
import time
from drudge import Tensor
from gristmill import optimize, get_flop_cost
from gristmill.generate import FortranPrinter
from sympy import IndexedBase
from sympy.printing.fortran import FCodePrinter

# Patch SymPy's Fortran printer:
# KroneckerDelta(i,j) -> deltaf(i,j)
def _print_KroneckerDelta(self, expr):
    i, j = expr.args
    return f"deltaf({self._print(i)}, {self._print(j)})"

FCodePrinter._print_KroneckerDelta = _print_KroneckerDelta

t0 = time.time()
Np = IndexedBase('Np')

expr1 = dr.define(Np[p], Ex_Np)
org_exp = [expr1]
 
print("Original Cost:", get_flop_cost(org_exp))
eval_equ0 = optimize(
    [expr1],
    interm_fmt='tau{}',
    drop_cutoff=2
)

print(type(eval_equ0))
 
print("Optimized cost:", get_flop_cost(eval_equ0))

 
fort = FortranPrinter()
code = fort.doprint(eval_equ0)
#print(code)


delta_func = """
pure double precision function deltaf(p, q)
  implicit none
  integer, intent(in) :: p, q

  if (p == q) then
    deltaf = 1.0d0
  else
    deltaf = 0.0d0
  end if

end function deltaf
"""

with open("BCS_CCSDTQ_Np.f90", "w") as fp:
    fp.write(delta_func)
    fp.write("\n\n")
    fp.write(code)

print("Printed!", time.time() - t0)


Original Cost: 20*na**4 + 20*na**3 + 15*na**2 + 30*na
<class 'list'>
Optimized cost: 6*na**4 + 6*na**3 + 7*na**2 + 23*na
Printed! 4.126438856124878
